In [1]:
#CookCash


In [2]:
import pandas as pd
import numpy as np
from pymongo import MongoClient
from sklearn.model_selection import train_test_split
import re # Library wajib buat manipulasi teks/kata

df = pd.read_csv('Book5.csv', sep=';')

print("Data mentah berhasil dimuat! Ukuran awal:", df.shape)
display(df.head(3))

Data mentah berhasil dimuat! Ukuran awal: (11595, 10)


,Title,Ingredients,Steps,Loves,URL,Category,Title Cleaned,Total Ingredients,Ingredients Cleaned,Total Steps
0,Ayam Woku Manado,1 Ekor Ayam Kampung (potong 12)--2 Buah Jeruk ...,1) Cuci bersih ayam dan tiriskan. Lalu peras j...,1,https://cookpad.com/id/resep/4473027-ayam-woku...,ayam,ayam woku manado,14,"ayam kampung potong , jeruk nipis , garam , ku...",7
1,Ayam goreng tulang lunak,1 kg ayam (dipotong sesuai selera jangan kecil...,"1) Haluskan bumbu2nya (BaPut, ketumbar, kemiri...",1,https://cookpad.com/id/resep/4471956-ayam-gore...,ayam,ayam goreng tulang lunak,11,"ayam dipotong , serai , daun jeruk , bawang pu...",5
2,Ayam cabai kawin,1/4 kg ayam--3 buah cabai hijau besar--7 buah ...,1) Panaskan minyak di dalam wajan. Setelah min...,2,https://cookpad.com/id/resep/4473057-ayam-caba...,ayam,ayam cabai kawin,10,"ayam , cabai hijau , cabai merah rawit , bawan...",3


In [3]:
print("--- 1. DATA UNDERSTANDING ---")
print("\nInfo Kolom dan Tipe Data:")
df.info()

print("\nCek Data Kosong (Missing Values):")
print(df.isnull().sum())

print("\nCek Data Duplikat:")
print("Jumlah duplikat:", df.duplicated().sum())

--- 1. DATA UNDERSTANDING ---

Info Kolom dan Tipe Data:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11595 entries, 0 to 11594
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Title                11595 non-null  object
 1   Ingredients          11595 non-null  object
 2   Steps                11595 non-null  object
 3   Loves                11595 non-null  int64 
 4   URL                  11595 non-null  object
 5   Category             11595 non-null  object
 6   Title Cleaned        11594 non-null  object
 7   Total Ingredients    11595 non-null  int64 
 8   Ingredients Cleaned  11595 non-null  object
 9   Total Steps          11595 non-null  int64 
dtypes: int64(3), object(7)
memory usage: 906.0+ KB

Cek Data Kosong (Missing Values):
Title                  0
Ingredients            0
Steps                  0
Loves                  0
URL                    0
Category               0
Title Cle

In [4]:
import pandas as pd
import re

print("--- 2. DATA CLEANING ---")

# 1. Hapus kolom yang tidak diperlukan
df.drop(columns=['URL', 'Title', 'Ingredients'], inplace=True, errors='ignore')

# 2. Tentukan kolom penting
kolom_wajib = ['Title Cleaned', 'Ingredients Cleaned', 'Steps', 'Category']

# Hapus missing values di awal agar proses teks lebih ringan
df.dropna(subset=kolom_wajib, inplace=True)

# =========================================================================
# 2.5: MEMBERSIHKAN NOISE DI JUDUL DAN BAHAN SECARA TOTAL
# =========================================================================

# Trik 1: Kompres huruf lebay / alay (Hanya yang diulang 3x atau lebih)
df['Title Cleaned'] = df['Title Cleaned'].str.replace(r'([a-zA-Z])\1{2,}', r'\1', regex=True)

# Trik 2: Sapu bersih karakter non-alfabet (Angka, Huruf Arab, Simbol, Emoji)
# Hanya menyisakan huruf a-z dan spasi pada Judul dan Bahan
df['Title Cleaned'] = df['Title Cleaned'].str.replace(r'[^a-zA-Z\s]', ' ', regex=True)
df['Ingredients Cleaned'] = df['Ingredients Cleaned'].str.replace(r'[^a-zA-Z\s]', ' ', regex=True)

# Trik 3: List kata buang (Sudah digabung dengan kata pengantar kreator)
kata_buang = [
    # Kata pengantar kreator
    'ala', 'by', 'dapur', 'dapoer', 'resep', 'versi', 'kreasi', 'ummu', 'khas',
    
    # Jargon Diet & Kesehatan
    'debm', 'keto', 'ketofastosis', 'diet', 'menu diet', 'no msg', 'msg',
    
    # Kata sifat, rasa, dan jargon lebay/alay beserta variasi typo-nya
    'simpel', 'simple', 'mudah', 'enak', 'enakk', 'uenak', 'ueenak', 'uenakk', 'ueenakk',
    'endes', 'endeuss', 'endues', 'endess', 'praktis', 'gampang', 'super', 'spesial', 
    'special', 'mantap', 'mantabb', 'sederhana', 'yummy', 'kriuk', 'anti ribet', 
    'murah meriah', 'ekonomis', 'nikmat', 'suka', 'nampol', 'anti gagal', 'sedap', 
    'maknyus', 'pedes', 'pedas', 'seger', 'segar', 'empuk', 'gurih', 'banget', 
    'krenyes', 'hepikol', 'pol', 'hai', 'lama', 'manjah', 'hot', 'empyuk', 'ekspres', 
    'enyak', 'nyem', 'cryes', 'syeger', 'yami', 'sadap', 'puedes', 'pwedes', 'mo', 
    'dor', 'ttt', 'akuh',
    
    # Istilah tidak penting & pelengkap
    'akuhhh', 'hihihi', 'kadarnya', 'niceketo', 'hemat gas', 'jogja banget', 
    'setan', 'menu balita', 'pengembang', 'pengenyal', 'masakan', 
    'alalala', 'resep kilat', 'yah', 'hehe', 'irit', 'without pho', 
    'rumahan', 'pemalas', 'ngamuk', 'fatsecret', 'rindu', 'ncc', 'nibbles', 
    'kesukaan', 'edisi', 'set', 'kawin', 'lepas', 'citarasa', 'murni', 'pnjang', 
    'sy', 'r', 'nr', 'cay', 'anak kos', 'anak kost', 'anak kosan', 'mertua',
    
    # Nama kreator / username / kata random spesifik
    'bunda', 'mama', 'umy', 'ovi', 'bu quro v', 'atha', 'rimara', 'vey', 'mamais', 'ph', 
    'dee', 'valen', 'slowakia', 'febs', 'diana', 'gerry', 'girianza', 'raffa', 
    'agnes', 'ade', 'asep', 'dina', 'juna', 'mela', 'suntoro', 'amih', 'fatih', 
    'lucy', 'linggar', 'kina', 'fitri', 'rasya', 'fe', 'deyarl', 'trias', 'erni', 
    'hartanti', 'qeela', 'rawid', 'jkb', 'les', 'berli', 'kirana', 'queen', 'uti', 
    'lila', 'kee', 'makban'
]

# Gabungkan list jadi satu pola regex besar & eksekusi penghapusan
pola_kata = r'\b(?:' + '|'.join(kata_buang) + r')\b'
df['Title Cleaned'] = df['Title Cleaned'].str.replace(pola_kata, '', flags=re.IGNORECASE, regex=True)

# Trik 4: Rapikan spasi yang dobel gara-gara ada kata/simbol yang dihapus
df['Title Cleaned'] = df['Title Cleaned'].str.replace(r'\s+', ' ', regex=True).str.strip()
df['Ingredients Cleaned'] = df['Ingredients Cleaned'].str.replace(r'\s+', ' ', regex=True).str.strip()
# =========================================================================

# 3. Hapus baris yang judulnya jadi kosong setelah di-cleaning 
df = df[df['Title Cleaned'].str.strip() != '']

print("\nMissing values setelah cleaning:")
print(df[kolom_wajib].isnull().sum())

# 4. Hapus Duplikat
print("\nJumlah duplikat sebelum:", df.duplicated().sum())
df.drop_duplicates(inplace=True)

# 5. Info akhir
print("\nJumlah duplikat setelah:", df.duplicated().sum())
print("Jumlah data setelah:", df.shape)
print("\nProses Data Cleaning Selesai!")

# Cek 15 data sampel untuk validasi akhir
display(df[['Title Cleaned', 'Ingredients Cleaned']].sample(15))

--- 2. DATA CLEANING ---

Missing values setelah cleaning:
Title Cleaned          0
Ingredients Cleaned    0
Steps                  0
Category               0
dtype: int64

Jumlah duplikat sebelum: 0

Jumlah duplikat setelah: 0
Jumlah data setelah: (11594, 7)

Proses Data Cleaning Selesai!


,Title Cleaned,Ingredients Cleaned
2728,sup kepala ikan kakap,kepala ikan kakap jahe bawang merah diiris teb...
4920,sop iga sapi,iga sapi wortel kentang bawang putih jahe lada...
8349,dadar telur cah tauge recipe dapurvy,telur kocok garam lada dadar bawang merah bawa...
6090,daging tahu,daging sapi tahu putih bumbu halus bawang mera...
6678,orak arik tahu kemangi,tahu kuning hancurkan dg tangan kemangi petik ...
4958,bihun goreng sapi,bihun jagung daging sapi bumbu mie goreng indo...
6396,tahu telur asin,tahu kuning telur asin pisahkan kuning putihny...
8936,tempe isi sambal,papan tempe tebal lg tengahnya jgn putus tepun...
4418,sapi lada hitam,daging sapi has kualitas bagus potong kotak ba...
1620,nila bakar manis,ikan nila segar gula merah ketumbar kemiri baw...


In [5]:
print("--- 5. DATA TRANSFORMATION ---")

# 1. Pastikan kolom numerik jadi Integer (Biar Chatbot bisa ngitung)
df['Loves'] = pd.to_numeric(df['Loves'], errors='coerce').fillna(0).astype(int)
df['Total Ingredients'] = pd.to_numeric(df['Total Ingredients'], errors='coerce').fillna(0).astype(int)
df['Total Steps'] = pd.to_numeric(df['Total Steps'], errors='coerce').fillna(0).astype(int)

# 2. Pastikan kolom teks jadi String
kolom_teks = ['Title Cleaned', 'Ingredients Cleaned', 'Steps', 'Category']

for col in kolom_teks:
    if col in df.columns: # Cek dulu biar gak error kalau kolomnya gak ada
        df[col] = df[col].astype(str)
        
# 3. REORDER: Susun ulang urutan kolom untuk mempermudah pembacaan
urutan_kolom = [
    'Title Cleaned', 
    'Ingredients Cleaned', 
    'Steps', 
    'Loves', 
    'Category', 
    'Total Ingredients', 
    'Total Steps'
]
df = df[urutan_kolom]

print("Transformasi Tipe Data & Reorder Kolom Selesai!")
display(df.head(2))
display(df.dtypes)

--- 5. DATA TRANSFORMATION ---
Transformasi Tipe Data & Reorder Kolom Selesai!


,Title Cleaned,Ingredients Cleaned,Steps,Loves,Category,Total Ingredients,Total Steps
0,ayam woku manado,ayam kampung potong jeruk nipis garam kunyit b...,1) Cuci bersih ayam dan tiriskan. Lalu peras j...,1,ayam,14,7
1,ayam goreng tulang lunak,ayam dipotong serai daun jeruk bawang putih ha...,"1) Haluskan bumbu2nya (BaPut, ketumbar, kemiri...",1,ayam,11,5


Title Cleaned          object
Ingredients Cleaned    object
Steps                  object
Loves                   int64
Category               object
Total Ingredients       int64
Total Steps             int64
dtype: object

In [6]:
from sklearn.model_selection import train_test_split

print("--- 6. DATA SPLITTING ---")

# Membagi data: 80% Train (Masuk DB), 20% Test 
df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)

print(f" Data berhasil dibagi!")
print(f"   - Data Train: {len(df_train)} baris (Akan dikirim ke MongoDB)")
print(f"   - Data Test : {len(df_test)} baris (Disimpan sebagai file backup)")

# Simpan Data Test ke CSV 
df_test.to_csv('Cookcash_Data_Test.csv', index=False)
print(" File 'Cookcash_Data_Test.csv' berhasil dibuat.")

--- 6. DATA SPLITTING ---
 Data berhasil dibagi!
   - Data Train: 9275 baris (Akan dikirim ke MongoDB)
   - Data Test : 2319 baris (Disimpan sebagai file backup)
 File 'Cookcash_Data_Test.csv' berhasil dibuat.


In [7]:
import pymongo
from pymongo import MongoClient

print("--- 7. EXPORT TO MONGODB ---")

try:
    # 1. Koneksi ke MongoDB Lokal
    client = MongoClient('mongodb://localhost:27017/')
    db = client['cookcash_db'] # Nama Database kamu
    collection = db['resep']   # Nama Koleksi (Laci) resep kamu

    # 2. Kosongkan data lama (supaya tidak duplikat kalau di-run ulang)
    collection.delete_many({})

    # 3. Ubah df_train ke format dictionary yang dipahami MongoDB
    data_dict = df_train.to_dict(orient='records')

    # 4. Masukkan data
    collection.insert_many(data_dict)
    
    print(f" SUKSES! {len(data_dict)} data resep sukses masuk ke MongoDB.")
    print("Silakan cek di MongoDB Compass pada database 'cookcash_db'.")

except Exception as e:
    print(f" Koneksi Gagal! Pastikan MongoDB Compass / Service sudah jalan.")
    print(f"Error: {e}")

--- 7. EXPORT TO MONGODB ---
 SUKSES! 9275 data resep sukses masuk ke MongoDB.
Silakan cek di MongoDB Compass pada database 'cookcash_db'.
